## 0) Setup / Imports


In [10]:
import pandas as pd
import re
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 1) Data Loading & Text Helpers


In [11]:
def load_dataset(path):
    df = pd.read_csv(path)
    return df

def load_and_merge_movies(movies_path, credits_path):
    movies = pd.read_csv(movies_path)
    credits = pd.read_csv(credits_path)

    # Normalize column names for merge
    if "movie_id" in credits.columns and "id" in movies.columns:
        merged = movies.merge(credits, left_on="id", right_on="movie_id", how="left")
    else:
        merged = movies.merge(credits, on="title", how="left")

    return merged

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    # keep letters + spaces only
    text = re.sub(r"[^a-z\s]", " ", text)
    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_names(value):
    """
    Handles multiple genre/keyword formats:
    - JSON-like list of dicts: '[{"id": 28, "name": "Action"}, ...]'
    - Python literal list of dicts: "[{'id': 16, 'name': 'Animation'}, ...]"
    - Pipe-separated: "Adventure|Comedy"
    - Comma-separated: "Action, Drama"
    """
    if pd.isna(value):
        return ""

    # If already a list, extract names directly
    if isinstance(value, list):
        names = []
        for item in value:
            if isinstance(item, dict):
                name = item.get("name", "")
                if name:
                    names.append(name)
            else:
                name = str(item).strip()
                if name:
                    names.append(name)
        return " ".join(names)

    text = str(value).strip()
    if not text:
        return ""

    # Try parsing list/dict literals
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            names = []
            for item in parsed:
                if isinstance(item, dict):
                    name = item.get("name", "")
                    if name:
                        names.append(name)
                else:
                    name = str(item).strip()
                    if name:
                        names.append(name)
            return " ".join(names)
        if isinstance(parsed, dict):
            name = parsed.get("name", "")
            return name if name else ""
    except Exception:
        pass

    # Fallback: split on pipe or comma
    if "|" in text:
        parts = [p.strip() for p in text.split("|")]
    else:
        parts = [p.strip() for p in text.split(",")]

    return " ".join([p for p in parts if p])


## 2) Text Preprocessing


In [12]:
def preprocess_data(df):
    """
    Build a combined text field for each movie.
    Uses whatever columns exist among:
    - title (or title_x after merge)
    - overview
    - genres
    - keywords
    - cast
    - crew
    """
    processed_text = []

    title_col = "title"
    if title_col not in df.columns and "title_x" in df.columns:
        title_col = "title_x"
    elif title_col not in df.columns and "title_y" in df.columns:
        title_col = "title_y"

    for i in range(len(df)):
        title = clean_text(df.loc[i, title_col]) if title_col in df.columns else ""
        overview = clean_text(df.loc[i, "overview"]) if "overview" in df.columns else ""
        genres = extract_names(df.loc[i, "genres"]) if "genres" in df.columns else ""
        keywords = extract_names(df.loc[i, "keywords"]) if "keywords" in df.columns else ""
        cast = extract_names(df.loc[i, "cast"]) if "cast" in df.columns else ""
        crew = extract_names(df.loc[i, "crew"]) if "crew" in df.columns else ""

        combined = f"{title} {overview} {genres} {keywords} {cast} {crew}"
        combined = clean_text(combined)

        processed_text.append(combined)

    return processed_text


## 3) TF-IDF Vectorization


In [13]:
def build_tfidf(text_data):
    if not any(t.strip() for t in text_data):
        raise ValueError(
            "No usable text found. Check that your dataset has columns like "
            "'title', 'overview', 'genres', or 'keywords', and that they contain text."
        )

    tfidf = TfidfVectorizer(
        stop_words="english",
        max_features=10000,
        ngram_range=(1, 2)
    )
    vectors = tfidf.fit_transform(text_data)
    return tfidf, vectors


## 4) User Input (text/genres)


In [14]:
def get_user_input():
    print("Enter your preferences (genre / themes / keywords).")
    print("Examples:")
    print("- space survival")
    print("- romance comedy")
    print("- crime investigation thriller")
    user_input = input("\nYour preference: ")
    return user_input

def process_user_input(user_input, tfidf):
    user_input = clean_text(user_input)
    user_vector = tfidf.transform([user_input])
    return user_vector

## 5) Cosine Similarity


In [15]:
def compute_similarity(user_vector, movie_vectors):
    similarity = cosine_similarity(user_vector, movie_vectors)
    return similarity


In [16]:
def rank_movies(similarity, df):
    scores = list(enumerate(similarity[0]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    title_col = "title"
    if title_col not in df.columns and "title_x" in df.columns:
        title_col = "title_x"
    elif title_col not in df.columns and "title_y" in df.columns:
        title_col = "title_y"

    ranked = []
    for idx, score in scores:
        if title_col in df.columns:
            title = df.iloc[idx][title_col]
        else:
            title = f"Movie {idx}"
        ranked.append({"title": title, "score": float(score)})
    return ranked


## 6) Ranking


In [17]:
def show_recommendations(recommendations, user_vector, similarity, user_input, top_n=5):
    print(f"\nYour preference: {user_input}")

    # If user input produced no known TF-IDF terms
    if user_vector.nnz == 0:
        print("\nNo recommendations: your input words were not found in the dataset vocabulary.")
        print("Try using genre/theme words like: action, romance, space, crime, thriller, comedy, etc.")
        return

    # If everything ties at 0 similarity
    if similarity.max() == 0:
        print("\nNo strong matches found for your input.")
        print("Try adding more keywords.")
        return

    print(f"\nTop {top_n} Recommendations:\n")
    count = 0
    for rec in recommendations:
        if rec["score"] > 0:
            count += 1
            print(f"{count}. {rec['title']}")
        if count == top_n:
            break

    if count == 0:
        print("No recommendations found. Try different keywords.")

## 6) Main


In [24]:
def main():
    movies_path = "imdb-dataset/tmdb_5000_movies.csv"
    credits_path = "imdb-dataset/tmdb_5000_credits.csv"

    df = load_and_merge_movies(movies_path, credits_path)
    text_data = preprocess_data(df)
    tfidf, movie_vectors = build_tfidf(text_data)

    user_input = get_user_input()
    user_vector = process_user_input(user_input, tfidf)

    similarity = compute_similarity(user_vector, movie_vectors)
    ranked_movies = rank_movies(similarity, df)

    show_recommendations(ranked_movies, user_vector, similarity, user_input, top_n=5)
    
if __name__ == "__main__":
    main()


Enter your preferences (genre / themes / keywords).
Examples:
- space survival
- romance comedy
- crime investigation thriller

Your preference: I want to see science fiction movie with mysterious plot

Top 5 Recommendations:

1. The Helix... Loaded
2. Ultramarines: A Warhammer 40,000 Movie
3. Flatliners
4. Queen of the Damned
5. Act of Valor
